# Retrain Model and Test Automated Pipeline

## Objectives
* Retrain the ML model after transferring custom transformer into src/feature_engineering.py and preprocessor under src/preprocessing.py
* Confirm that automated pipelines work without breaking.
* Re-predict inherited house prices and confirm predictions remain unchanged.  

## Inputs
* `house_prices_records.csv` data located at outputs/datasets/collection
* `inherited_houses.csv` data located at outputs/datasets/collection. This is the dataset on customer's inherited houses' characteristics.

## Outputs
* Automated ML pipelines.
* Saved model under outputs/models/house_price_pipeline.joblib
* A saved fitted pipeline for deployment in the Streamlit dashboard.

The two previous notebooks (`05_Modelling_and_ML_Pipeline` and `06_Inherited_House_Price_Predictions`) trained the model, selected the best performing option and predicted inherited house prices. This notebook is designed 
- to retrain the model after transferring custom transformer into src/feature_engineering.py and the preprocessor into src/preprocessing.py and 
- to confirm that automated pipelines work without issues.   

## Change Working Directory

In [3]:
import os
from pathlib import Path

project_root = Path.cwd()
# Move up until we find the project root (identified by a folder/file as marker)
while not (project_root / ".git").exists():
    project_root = project_root.parent

os.chdir(project_root)

print("Working directory set to:", project_root)

Working directory set to: /Users/mehtap/Documents/GitHub/heritage-housing-issues


## Imports and Setup

In [16]:
from src.feature_engineering import FeatureEngineerHPData
from src.preprocessing import create_preprocessor

import numpy as np
import pandas as pd
import joblib

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
# from sklearn.compose import ColumnTransformer
# from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
# from sklearn.impute import SimpleImputer
# from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
# from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor

## Load Data

Load the `house_prices_records` and `inherited_houses` data.

In [5]:
df = pd.read_csv('outputs/datasets/collection/house_prices_records.csv')
df.head()

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,SalePrice
0,856,854.0,3.0,No,706,GLQ,150,0.0,548,RFn,...,65.0,196.0,61,5,7,856,0.0,2003,2003,208500
1,1262,0.0,3.0,Gd,978,ALQ,284,NaN,460,RFn,...,80.0,0.0,0,8,6,1262,NaN,1976,1976,181500
2,920,866.0,3.0,Mn,486,GLQ,434,0.0,608,RFn,...,68.0,162.0,42,5,7,920,NaN,2001,2002,223500
3,961,NaN,NaN,No,216,ALQ,540,NaN,642,Unf,...,60.0,0.0,35,5,7,756,NaN,1915,1970,140000
4,1145,NaN,4.0,Av,655,GLQ,490,0.0,836,RFn,...,84.0,350.0,84,5,8,1145,NaN,2000,2000,250000


In [6]:
df_predict = pd.read_csv("outputs/datasets/collection/inherited_houses.csv")
df_predict.head()

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotArea,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd
0,896,0,2,No,468.0,Rec,270.0,0,730.0,Unf,...,11622,80.0,0.0,0,6,5,882.0,140,1961,1961
1,1329,0,3,No,923.0,ALQ,406.0,0,312.0,Unf,...,14267,81.0,108.0,36,6,6,1329.0,393,1958,1958
2,928,701,3,No,791.0,GLQ,137.0,0,482.0,Fin,...,13830,74.0,0.0,34,5,5,928.0,212,1997,1998
3,926,678,3,No,602.0,GLQ,324.0,0,470.0,Fin,...,9978,78.0,20.0,36,6,6,926.0,360,1998,1998


## Define Target and Predictors

In [7]:
target_variable = 'SalePrice'
X = df.drop(columns=[target_variable]).copy()
y = np.log(df[target_variable]).copy()

## Split Data into Train and Test Samples

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=34
)

print(X_train.shape, X_test.shape)

(1168, 23) (292, 23)


## Define Variable Groups

In [11]:
log_vars = ["1stFlrSF", "GrLivArea", "LotArea"]

log1p_vars = [
    "BsmtFinSF1",
    "BsmtUnfSF",
    "TotalBsmtSF",
    "GarageArea",
    "LotFrontage",
    "MasVnrArea"
]

numeric_and_binary_vars = [
    "2ndFlrSF",
    "GarageYrBlt",
    "BedroomAbvGr",
    "EnclosedPorch",
    "OpenPorchSF",
    "WoodDeckSF",
    "OverallCond",
    "OverallQual",
    "YearBuilt",
    "YearRemodAdd",
    "TotalPorchSF",
    "MissingLotFrontage",
    "MissingMasVnrArea",
    "MissingGarageYrBlt",
    "MissingBedroomAbvGr",
    "MissingBsmtFinType1",
    "Has2ndFlr",
    "HasExtraLivArea",
    "HasBasement",
    "HasBsmtFin",
    "HasBsmtUnf",
    "HasGarage",
    "HasLargeLotArea",
    "HasSmallLotArea",
    "HasMasVnr",
    "HasEnclosedPorch",
    "HasOpenPorch",
    "HasWoodDeck",
    "BuiltPre1950"
]

categorical_vars = [
    "BsmtExposure",
    "BsmtFinType1",
    "GarageFinish",
    "KitchenQual"
]

In [17]:
preprocessor = create_preprocessor(
    log_vars=log_vars,
    log1p_vars=log1p_vars,
    numeric_and_binary_vars=numeric_and_binary_vars,
    categorical_vars=categorical_vars,
)

rf_pipeline = Pipeline(steps=[
    ("feature_engineering", FeatureEngineerHPData(bedroom_impute_strategy="mean")),
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=34))
])

In [18]:
rf_param_grid = {
    "model__n_estimators": [200, 500],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

rf_search = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=rf_param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

rf_search.fit(X_train, y_train)

print("Best RF parameters:", rf_search.best_params_)
print("Best RF CV score:", rf_search.best_score_)

Best RF parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best RF CV score: 0.8649963104805998


In [20]:
best_rf = rf_search.best_estimator_
joblib.dump(best_rf, "outputs/models/house_price_pipeline.joblib")

['outputs/models/house_price_pipeline.joblib']

## Confirm Predictions 

In [21]:
loaded_model = joblib.load("outputs/models/house_price_pipeline.joblib")

In [22]:
log_price_predictions = loaded_model.predict(df_predict)
price_predictions = np.exp(log_price_predictions)

df_predict_results = df_predict.copy()
df_predict_results["PredictedSalePrice"] = price_predictions

df_predict_results

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,PredictedSalePrice
0,896,0,2,No,468.0,Rec,270.0,0,730.0,Unf,...,80.0,0.0,0,6,5,882.0,140,1961,1961,124407.053598
1,1329,0,3,No,923.0,ALQ,406.0,0,312.0,Unf,...,81.0,108.0,36,6,6,1329.0,393,1958,1958,157170.742599
2,928,701,3,No,791.0,GLQ,137.0,0,482.0,Fin,...,74.0,0.0,34,5,5,928.0,212,1997,1998,169290.139425
3,926,678,3,No,602.0,GLQ,324.0,0,470.0,Fin,...,78.0,20.0,36,6,6,926.0,360,1998,1998,185306.232258


Round to two decimal points

In [23]:
df_predict_results["PredictedSalePrice"] = df_predict_results["PredictedSalePrice"].round(2)
df_predict_results

,1stFlrSF,2ndFlrSF,BedroomAbvGr,BsmtExposure,BsmtFinSF1,BsmtFinType1,BsmtUnfSF,EnclosedPorch,GarageArea,GarageFinish,...,LotFrontage,MasVnrArea,OpenPorchSF,OverallCond,OverallQual,TotalBsmtSF,WoodDeckSF,YearBuilt,YearRemodAdd,PredictedSalePrice
0,896,0,2,No,468.0,Rec,270.0,0,730.0,Unf,...,80.0,0.0,0,6,5,882.0,140,1961,1961,124407.05
1,1329,0,3,No,923.0,ALQ,406.0,0,312.0,Unf,...,81.0,108.0,36,6,6,1329.0,393,1958,1958,157170.74
2,928,701,3,No,791.0,GLQ,137.0,0,482.0,Fin,...,74.0,0.0,34,5,5,928.0,212,1997,1998,169290.14
3,926,678,3,No,602.0,GLQ,324.0,0,470.0,Fin,...,78.0,20.0,36,6,6,926.0,360,1998,1998,185306.23


Save predictions

In [24]:
df_predict_results.to_csv("outputs/datasets/predictions/predicted_house_prices.csv", index=False)
print("Prediction file saved.")

Prediction file saved.


Sum of predicted prices for all four inherited houses 

In [25]:
df_predict_results["PredictedSalePrice"].sum()

636174.16

## Conclusions

The custom transformer and the processor is integrated into automated pipeline without any issues. 

## Next Steps

Integrating the fitted pipeline into Streamlit Dashboard.